In [ ]:
import datetime, subprocess, torch
print(datetime.datetime.now(datetime.timezone.utc).isoformat())
print('CUDA:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

2026-11-11T00:34:22.053062+00:00
CUDA: True
Wed Nov 11 00:34:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---

In [ ]:
%pip install -q 'trainer-cli[train]==0.77.2' 'transformers<5' datasets bitsandbytes
!trainer --version
!nvidia-smi

ERROR: Cannot install trainer-cli[train]==0.77.2 and transformers<5 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
/bin/bash: line 1: trainer: command not found
Wed Nov 11 00:34:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tes

In [ ]:
!trainer data doctor data/train.jsonl --model Qwen/Qwen2.5-1.5B-Instruct --format alpaca --max-length 512 --sample 200 --show-mask 2 --output data-doctor.json
!trainer doctor --config trainer.yaml
!trainer plan --config trainer.yaml

config.json: 100% 660/660 [00:00<00:00, 3.87MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 29.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 68.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 113MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 139MB/s]
[transformers] return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.
                                trainer data doctor                                
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check              ┃ Verdict ┃ Message               ┃ Evidence              ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ chat_template      │ OK      │ tokenizer has a       │ 2507 chars            │
│                    │         │ chat_template         │                       │
│ template_render    │ OK      │ 0/200 rows failed to  │                       │
│                    │         │ render th

In [ ]:
%pip install -q 'trainer-cli[train]==0.77.2' datasets bitsandbytes
!trainer --version
import transformers, torch
print('transformers', transformers.__version__, 'torch', torch.__version__)

trainer v0.77.2
https://github.com/MakazhanAlpamys/Trainer
transformers 5.22.1 torch 2.14.1+cu130


In [ ]:
from datasets import load_dataset
import json, hashlib, pathlib
rows = load_dataset('databricks/databricks-dolly-15k', split='train').shuffle(seed=20261111).select(range(1500))
out = [{'instruction': r['instruction'], 'input': r['context'] or '', 'output': r['response']} for r in rows]
assert len(out) == 1500 and all(x['instruction'].strip() and x['output'].strip() for x in out)
assert len(set(hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest() for x in out)) == 1500
pathlib.Path('data').mkdir(exist_ok=True)
for name, subset in [('train.jsonl', out[:1400]), ('heldout.jsonl', out[1400:])]:
    p = pathlib.Path('data') / name
    p.write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in subset))
    print(name, len(subset), hashlib.sha256(p.read_bytes()).hexdigest())
print('sample', json.dumps(out[0], ensure_ascii=False)[:1000])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

train.jsonl 1400 0e148de110780f7a1e5c06ccee6a4a5138f73c2dc0ce04770438b6470769fc26
heldout.jsonl 100 ac218ecb3981b53bb550f7278ece374b084f04f50681f60d369b89d8c2e79f29
sample {"instruction": "How to prepare without baking dessert recipe -Banana Pudding?", "input": "", "output": "Take a casserole mold. Layer the mold's base with cookies of your choice. Slice up 1½ - 2 bananas and layer them above the cookies. The pudding mixture can be made by following the instructions on the box. Mix the tub of Cool Whip well before pouring it above the pudding mixture. Pour half of the pudding mix on the first layer of bananas and cookies. Add more bananas and cookies, followed by more pudding mixture. Repeat the process to add more layers. Use cookie crumbs and sprinkle on top. Refrigerate until serving."}


In [ ]:
%%writefile trainer.yaml
base: Qwen/Qwen2.5-1.5B-Instruct
task: sft
data:
  train: ./data/train.jsonl
  format: alpaca
  val_split: 0.1
  max_length: 512
training:
  epochs: 1
  lr: 0.0002
  batch_size: 1
  lora:
    r: 16
    alpha: 32
    target_modules: [q_proj, v_proj]
  quantization: 4bit
output: ./output

Writing trainer.yaml


In [ ]:
import subprocess, threading, datetime, time, pathlib, sys
def stamp(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
for cmd in [['trainer','doctor','--config','trainer.yaml'], ['trainer','plan','--config','trainer.yaml']]:
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(stamp(), ' '.join(cmd), 'exit', p.returncode, p.stdout[-2000:], p.stderr[-1000:])
    assert p.returncode == 0, 'preflight failed'
stop = threading.Event()
def monitor():
    with open('raw_nvidia-smi_colab.txt','w') as f:
        while not stop.is_set():
            f.write(stamp()+'\n'+subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout+'\n')
            f.flush()
            stop.wait(2)
t = threading.Thread(target=monitor,daemon=True); t.start()
try:
    with open('raw_train_colab.log','w') as f:
        p = subprocess.Popen(['trainer','train','--config','trainer.yaml','--yes'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            record = stamp()+' '+line
            f.write(record); f.flush(); print(record, end='')
        rc = p.wait()
finally:
    stop.set(); t.join()
print('TRAIN EXIT', rc, 'at', stamp())
assert rc == 0

2026-11-11T00:37:56.167229+00:00 trainer doctor --config trainer.yaml exit 0 tional │   0.141.1    │ >=0.104.0   │ OK                    │
│ uvicorn      │ optional │    0.54.0    │ >=0.24.0    │ OK                    │
│ datasketch   │ optional │      -       │ >=1.6.0     │ not installed         │
│ lm-eval      │ optional │      -       │ >=0.4.0     │ not installed         │
│ wandb        │ optional │    0.28.1    │ >=0.15.0    │ OK                    │
│ deepspeed    │ optional │      -       │ >=0.12.0    │ not installed         │
│ httpx        │ optional │    0.28.1    │ >=0.24.0    │ OK                    │
│ unsloth      │ optional │      -       │ >=2024.8    │ not installed         │
│ Pillow       │ optional │    11.3.0    │ >=9.0.0     │ OK                    │
│ torchao      │ optional │    0.10.0    │ >=0.18.0    │ outdated (need        │
│              │          │              │             │ >=0.18.0)             │
│ sglang       │ optional │      -       │ >=0.5.1 

AssertionError: 

In [ ]:
from pathlib import Path
print('\n'.join(x[:200] for x in Path('raw_train_colab.log').read_text().splitlines()[-6:]))

2026-11-11T00:38:56.642358+00:00 Map:   0%|          | 3/1260 [00:00<00:15, 81.78 examples/s]
2026-11-11T00:38:56.764912+00:00 
2026-11-11T00:38:56.764981+00:00 Error: ValueError: train row 4: no causal-loss target remains after 
2026-11-11T00:38:56.765002+00:00 tokenization/truncation at data.max_length=512; the assistant response is absent
2026-11-11T00:38:56.765018+00:00 or fully truncated. Increase data.max_length or shorten the prompt.
2026-11-11T00:38:56.765358+00:00 Run with --verbose for the full traceback.


In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset
import json, hashlib, pathlib
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
source = load_dataset('databricks/databricks-dolly-15k', split='train').shuffle(seed=20261111)
short = []
seen = set()
for r in source:
    x = {'instruction': r['instruction'], 'input': r['context'] or '', 'output': r['response']}
    if not x['instruction'].strip() or not x['output'].strip(): continue
    h = hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()
    if h in seen: continue
    prompt = x['instruction'] + ('\n\n' + x['input'] if x['input'] else '')
    rendered = tok.apply_chat_template([{'role':'user','content':prompt},{'role':'assistant','content':x['output']}], tokenize=True)
    if len(rendered) > 480: continue
    short.append(x); seen.add(h)
    if len(short) == 1500: break
assert len(short) == 1500
for name, subset in [('train.jsonl', short[:1400]), ('heldout.jsonl', short[1400:])]:
    p = pathlib.Path('data') / name
    p.write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in subset))
    print(name, len(subset), hashlib.sha256(p.read_bytes()).hexdigest())
print('filter passed, longest rendered tokens', max(len(tok.apply_chat_template([{'role':'user','content':x['instruction']+'\n\n'+x['input']},{'role':'assistant','content':x['output']}], tokenize=True)) for x in short))

train.jsonl 1400 0e148de110780f7a1e5c06ccee6a4a5138f73c2dc0ce04770438b6470769fc26
heldout.jsonl 100 ac218ecb3981b53bb550f7278ece374b084f04f50681f60d369b89d8c2e79f29
filter passed, longest rendered tokens 2


In [ ]:
short = []
seen = set()
for r in source:
    x = {'instruction': r['instruction'], 'input': r['context'] or '', 'output': r['response']}
    if not x['instruction'].strip() or not x['output'].strip(): continue
    h = hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()
    if h in seen: continue
    prompt = x['instruction'] + ('\n\n' + x['input'] if x['input'] else '')
    rendered = tok.apply_chat_template([{'role':'user','content':prompt},{'role':'assistant','content':x['output']}], tokenize=True)
    ids = rendered['input_ids'] if isinstance(rendered, dict) else rendered
    if len(ids) > 480: continue
    short.append(x); seen.add(h)
    if len(short) == 1500: break
assert len(short) == 1500
for name, subset in [('train.jsonl', short[:1400]), ('heldout.jsonl', short[1400:])]:
    p = pathlib.Path('data') / name
    p.write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in subset))
    print(name, len(subset), hashlib.sha256(p.read_bytes()).hexdigest())
print('longest accepted tokens', max(len(tok.apply_chat_template([{'role':'user','content':x['instruction']},{'role':'assistant','content':x['output']}], tokenize=True)['input_ids']) for x in short))

train.jsonl 1400 0e148de110780f7a1e5c06ccee6a4a5138f73c2dc0ce04770438b6470769fc26
heldout.jsonl 100 ac218ecb3981b53bb550f7278ece374b084f04f50681f60d369b89d8c2e79f29
longest accepted tokens 2013


In [ ]:
r = source[0]
a = tok.apply_chat_template([{'role':'user','content':r['instruction']},{'role':'assistant','content':r['response']}], tokenize=True)
print(type(a), list(a) if isinstance(a,dict) else 'list', type(a['input_ids']) if isinstance(a,dict) else None)
print('len ids', len(a['input_ids']), 'first type', type(a['input_ids'][0]), 'first len', len(a['input_ids'][0]) if hasattr(a['input_ids'][0],'__len__') else None)
print('full with context', len(tok.apply_chat_template([{'role':'user','content':r['instruction']+'\n\n'+r['context']},{'role':'assistant','content':r['response']}], tokenize=True)['input_ids']))

<class 'transformers.tokenization_utils_base.BatchEncoding'> list None
len ids 156 first type <class 'int'> first len None
full with context 156


In [ ]:
short = []
seen = set()
for r in source:
    x = {'instruction': r['instruction'], 'input': r['context'] or '', 'output': r['response']}
    if not x['instruction'].strip() or not x['output'].strip(): continue
    h = hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()
    if h in seen: continue
    prompt = x['instruction'] + ('\n\n' + x['input'] if x['input'] else '')
    n = len(tok.apply_chat_template([{'role':'user','content':prompt},{'role':'assistant','content':x['output']}], tokenize=True)['input_ids'])
    if n > 480: continue
    short.append(x); seen.add(h)
    if len(short) == 1500: break
assert len(short) == 1500
for name, subset in [('train.jsonl', short[:1400]), ('heldout.jsonl', short[1400:])]:
    p = pathlib.Path('data') / name
    p.write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in subset))
    print(name, len(subset), hashlib.sha256(p.read_bytes()).hexdigest())
print('first length', len(tok.apply_chat_template([{'role':'user','content':short[0]['instruction']},{'role':'assistant','content':short[0]['output']}], tokenize=True)['input_ids']))

train.jsonl 1400 f2851c4d747445d727e86fd03483bfce30250b0843db25b38f8a86d3a4aeaab3
heldout.jsonl 100 87fb5abbb936b05909856c6f22ec56fdc84d106b0d737aee3e0d49b60bcf1002
first length 156


In [ ]:
import subprocess, threading, datetime
from pathlib import Path
def stamp(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
d = subprocess.run(['trainer','data','doctor','data/train.jsonl','--model','Qwen/Qwen2.5-1.5B-Instruct','--format','alpaca','--max-length','512','--sample','200','--output','data-doctor-v2.json'],capture_output=True,text=True)
Path('data-doctor-v2.log').write_text(d.stdout+d.stderr)
print(stamp(), 'data doctor exit', d.returncode, 'tail', (d.stdout+d.stderr)[-1200:])
assert d.returncode == 0
stop = threading.Event()
def monitor():
    with open('raw_nvidia-smi_v2.txt','w') as f:
        while not stop.is_set():
            f.write(stamp()+'\n'+subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout+'\n'); f.flush(); stop.wait(2)
t = threading.Thread(target=monitor,daemon=True); t.start()
try:
    with open('raw_train_v2.log','w') as f:
        p = subprocess.Popen(['trainer','train','--config','trainer.yaml','--yes'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            record=stamp()+' '+line; f.write(record); f.flush(); print(record,end='')
        rc=p.wait()
finally:
    stop.set(); t.join()
print('TRAIN V2 EXIT',rc,stamp())
assert rc==0

2026-11-11T00:42:30.602916+00:00 data doctor exit 0 tail es      │ OK      │ 0/200 rows (0.0%)     │                       │
│                    │         │ contain a message     │                       │
│                    │         │ role outside          │                       │
│                    │         │ ['assistant',         │                       │
│                    │         │ 'function', 'system', │                       │
│                    │         │ 'tool', 'user']       │                       │
│ truncation_risk    │ OK      │ p95 length = 425      │ p50=130, max=467      │
│                    │         │ tokens vs             │                       │
│                    │         │ max_length=512; 0/200 │                       │
│                    │         │ rows (0.0%) would be  │                       │
│                    │         │ truncated             │                       │
└────────────────────┴─────────┴───────────────────────┴─────────

In [ ]:
import subprocess, threading, datetime
def stamp(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
stop_merge = threading.Event()
def sample_merge():
    with open('raw_nvidia-smi_merge.txt','w') as f:
        while not stop_merge.is_set():
            f.write(stamp()+'\n'+subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout+'\n'); f.flush(); stop_merge.wait(1)
tm=threading.Thread(target=sample_merge,daemon=True); tm.start()
try:
    with open('raw_merge.log','w') as f:
        p=subprocess.Popen(['trainer','merge','--adapter','output','--base','Qwen/Qwen2.5-1.5B-Instruct','--output','merged','--dtype','float16','--low-memory'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in p.stdout:
            record=stamp()+' '+line; f.write(record); f.flush(); print(record,end='')
        rc=p.wait()
finally:
    stop_merge.set(); tm.join()
print('MERGE EXIT',rc,stamp())
assert rc==0

2026-11-11T00:49:33.786848+00:00 ╭───────────────────────────────── Merge Plan ─────────────────────────────────╮
2026-11-11T00:49:33.787047+00:00 │ Adapter: output                                                              │
2026-11-11T00:49:33.787092+00:00 │ Base:    Qwen/Qwen2.5-1.5B-Instruct                                          │
2026-11-11T00:49:33.787112+00:00 │ Output:  merged                                                              │
2026-11-11T00:49:33.787131+00:00 │ Dtype:   float16                                                             │
2026-11-11T00:49:33.787145+00:00 ╰──────────────────────────────────────────────────────────────────────────────╯
2026-11-11T00:49:39.855593+00:00 Base revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306 (recorded run)
2026-11-11T00:49:39.858485+00:00 Fetching base weights: Qwen/Qwen2.5-1.5B-Instruct...
2026-11-11T00:49:40.109580+00:00 
2026-11-11T00:49:40.114263+00:00 Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]
202

AssertionError: 

In [ ]:
import subprocess, threading, datetime
def stamp(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
stop_merge2=threading.Event()
def sample_merge2():
    with open('raw_nvidia-smi_merge_v2.txt','w') as f:
        while not stop_merge2.is_set():
            f.write(stamp()+'\n'+subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout+'\n'); f.flush(); stop_merge2.wait(1)
tm2=threading.Thread(target=sample_merge2,daemon=True); tm2.start()
try:
    with open('raw_merge_v2.log','w') as f:
        p=subprocess.Popen(['trainer','merge','--adapter','output','--base','Qwen/Qwen2.5-1.5B-Instruct','--output','merged','--dtype','float16'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in p.stdout:
            record=stamp()+' '+line; f.write(record); f.flush(); print(record,end='')
        rc=p.wait()
finally:
    stop_merge2.set(); tm2.join()
print('MERGE V2 EXIT',rc,stamp())
assert rc==0

2026-11-11T00:50:45.495175+00:00 ╭───────────────────────────────── Merge Plan ─────────────────────────────────╮
2026-11-11T00:50:45.495330+00:00 │ Adapter: output                                                              │
2026-11-11T00:50:45.495702+00:00 │ Base:    Qwen/Qwen2.5-1.5B-Instruct                                          │
2026-11-11T00:50:45.495739+00:00 │ Output:  merged                                                              │
2026-11-11T00:50:45.495757+00:00 │ Dtype:   float16                                                             │
2026-11-11T00:50:45.495771+00:00 ╰──────────────────────────────────────────────────────────────────────────────╯
2026-11-11T00:50:55.538405+00:00 Loading base model: Qwen/Qwen2.5-1.5B-Instruct...
2026-11-11T00:50:55.539221+00:00 Base revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306 (recorded run)
2026-11-11T00:50:55.754011+00:00 
2026-11-11T00:50:56.598042+00:00 Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]
2026-

AssertionError: 

In [18]:
import subprocess; rc=subprocess.run('date -u +%FT%TZ > raw_quantize.log; trainer merge --adapter output --base Qwen/Qwen2.5-1.5B-Instruct --output quantized --save-format 4bit >> raw_quantize.log 2>&1',shell=True).returncode; print('QUANT_EXIT',rc)

KeyboardInterrupt: 

In [19]:
from pathlib import Path
import json, subprocess
for d in ['merged','quantized','quantized-final']:
    p=Path(d); print(d, p.exists(), [(x.name,x.stat().st_size) for x in p.glob('*') if x.is_file()][:10])
    if (p/'config.json').exists(): print('quantization', json.loads((p/'config.json').read_text()).get('quantization_config'))
print('quant log tail', Path('raw_quantize.log').read_text()[-2000:])

merged True [('generation_config.json', 242), ('tokenizer.json', 11421892), ('chat_template.jinja', 2507), ('tokenizer_config.json', 693), ('config.json', 1372), ('model.safetensors', 3087466808)]
quantization None
quantized True [('generation_config.json', 242), ('tokenizer.json', 11421892), ('chat_template.jinja', 2507), ('tokenizer_config.json', 693), ('config.json', 1852), ('model.safetensors', 1143327349)]
quantization {'_load_in_4bit': True, '_load_in_8bit': False, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': None, 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}
quantized-final False []
quant log tail s]
Loading weights: 100%|██████████| 338/338 [00:09<00:00, 34.16it/s]

Writing model shards:   0%|          | 0/1 [00:33<?, ?it/s]



In [20]:
%%writefile verify_artifacts.py
import gc, json
from pathlib import Path
import torch
from huggingface_hub import snapshot_download
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer

def tensor(folder, key):
    for file in Path(folder).glob('*.safetensors'):
        with safe_open(file, framework='pt', device='cpu') as sf:
            if key in sf.keys(): return sf.get_tensor(key).float()
    raise KeyError(key)

base='Qwen/Qwen2.5-1.5B-Instruct'
base_dir=snapshot_download(base, allow_patterns=['*.safetensors','*.json'])
key='model.layers.0.self_attn.q_proj.weight'
a=tensor(base_dir,key); b=tensor('merged',key)
delta=((b-a).norm()/a.norm()).item()
with safe_open('output/adapter_model.safetensors',framework='pt',device='cpu') as sf:
    bs=[sf.get_tensor(k).float().norm().item() for k in sf.keys() if 'lora_B' in k]
qcfg=json.loads(Path('quantized/config.json').read_text()).get('quantization_config',{})
rows=[json.loads(x) for x in Path('data/heldout.jsonl').read_text().splitlines()[:8]]
tok=AutoTokenizer.from_pretrained(base)
result={'base_revision':Path(base_dir).name,'relative_q_proj_delta':delta,'adapter_B_nonzero':sum(x>0 for x in bs),'adapter_B_count':len(bs),'merged_has_adapter_file':Path('merged/adapter_model.safetensors').exists(),'quantization_config':qcfg,'artifacts':{}}
for name,path in [('base',base),('merged','merged'),('4bit','quantized')]:
    model=AutoModelForCausalLM.from_pretrained(path,torch_dtype=torch.float16,device_map='auto').eval()
    total=count=0; answers=[]
    with torch.inference_mode():
        for row in rows:
            user=row['instruction']+('\n'+row['input'] if row.get('input') else '')
            prompt=tok.apply_chat_template([{'role':'user','content':user}],tokenize=True,add_generation_prompt=True)['input_ids']
            full=tok.apply_chat_template([{'role':'user','content':user},{'role':'assistant','content':row['output']}],tokenize=True)['input_ids']
            assert full[:len(prompt)]==prompt and len(prompt)<len(full)<=512
            ids=torch.tensor([full],device=model.device)
            logits=model(ids).logits[:,len(prompt)-1:-1].float()
            target=ids[:,len(prompt):]
            total+=torch.nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),target.reshape(-1),reduction='sum').item()
            count+=target.numel()
        for question in ['Why is a bicycle helmet useful?','List three steps for washing hands.']:
            ids=tok.apply_chat_template([{'role':'user','content':question}],tokenize=True,add_generation_prompt=True,return_tensors='pt')['input_ids'].to(model.device)
            answer=model.generate(ids,do_sample=False,max_new_tokens=64,pad_token_id=tok.eos_token_id)
            answers.append(tok.decode(answer[0,ids.shape[1]:],skip_special_tokens=True))
    result['artifacts'][name]={'heldout_nll_per_target_token':total/count,'target_tokens':count,'answers':answers}
    print(name,json.dumps(result['artifacts'][name]),flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
result['structural_pass']=delta>1e-6 and len(bs)>0 and all(x>0 for x in bs) and not result['merged_has_adapter_file']
result['quantized_close_nll']=abs(result['artifacts']['4bit']['heldout_nll_per_target_token']-result['artifacts']['merged']['heldout_nll_per_target_token'])<0.1
Path('verification.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result,indent=2))
assert result['structural_pass'] and result['quantized_close_nll']

Writing verify_artifacts.py


In [21]:
import subprocess, datetime
def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
with open('raw_verification.log','w') as f:
    f.write(now()+' start\n'); f.flush()
    rc=subprocess.run(['python','-u','verify_artifacts.py'],stdout=f,stderr=subprocess.STDOUT).returncode
    f.write(now()+f' exit {rc}\n')
print('VERIFY_EXIT',rc,now()); print(open('raw_verification.log').read()[-6000:])

VERIFY_EXIT 0 2026-11-11T01:09:16.581163+00:00
Loading weights: 100%|██████████| 338/338 [00:12<00:00, 27.71it/s]
merged {"heldout_nll_per_target_token": 2.351804429383977, "target_tokens": 457, "answers": ["A bicycle helmet protects the head from injury in case of an accident. It can also prevent serious injuries such as concussions and skull fractures.", "1. Wet your hands with clean running water.\n2. Apply soap and lather up the entire surface of both hands.\n3. Scrub all surfaces of both hands including between fingers and under nails."]}

Loading weights: 100%|██████████| 338/338 [00:04<00:00, 84.03it/s] 
4bit {"heldout_nll_per_target_token": 2.3785898633284956, "target_tokens": 457, "answers": ["A bicycle helmet protects your head from injury in the event of an accident. It can also protect you from serious injuries if you fall off your bike and hit your head on the ground.", "1. Wet your hands with clean running water.\n2. Apply soap and lather up the entire surface of your han

In [23]:
import json,re; from pathlib import Path
v=json.loads(Path('verification.json').read_text())
print([(k,round(x['heldout_nll_per_target_token'],4),x['target_tokens']) for k,x in v['artifacts'].items()]); print('qclose',v['quantized_close_nll'])
for name in ['v2','merge_v3','quantize']:
 p=Path('raw_nvidia-smi_'+name+'.txt'); vals=[int(x) for x in re.findall(r'(\d+)MiB\s*/\s*15360MiB',p.read_text())] if p.exists() else []; print(name,len(vals),max(vals) if vals else None)

[('base', 2.5135, 457), ('merged', 2.3518, 457), ('4bit', 2.3786, 457)]
qclose True
v2 175 5395
merge_v3 44 115
quantize 59 1257


In [ ]:
import json,subprocess,datetime
from pathlib import Path
tasks=[('What is 2 + 2?','4'),('What protects your head when cycling?','helmet'),('Name the planet we live on.','Earth'),('How many days are in a week?','7'),('What should you do before eating to clean your hands?','wash'),('What is frozen water called?','ice')]
Path('ship_tasks.jsonl').write_text(''.join(json.dumps({'prompt':q,'expected':a,'scoring':'contains'})+'\n' for q,a in tasks))
with open('raw_ship.log','w') as f:
 f.write(datetime.datetime.now(datetime.timezone.utc).isoformat()+' start\n'); f.flush()
 rc=subprocess.run(['trainer','ship','--base','Qwen/Qwen2.5-1.5B-Instruct','--adapter','output','--task-eval','ship_tasks.jsonl','--output','ship_verdict.json','--device','cuda'],stdout=f,stderr=subprocess.STDOUT).returncode
 f.write(datetime.datetime.now(datetime.timezone.utc).isoformat()+f' exit {rc}\n')
print('SHIP_EXIT',rc); print(Path('raw_ship.log').read_text()[-2500:])

## Retry evidence and provenance

This local notebook export was saved before the second Colab VM finished. It preserves the first VM's failed and successful cells; it does not pretend that retry outputs were present in this export. The second VM reran the corrected data selection, training, merge, NF4 export, verification, and `trainer ship`. Exact raw outputs are in `retry_evidence/`. The gate decision is `DON'T SHIP`: task 1.00→1.00 (no strict win), mini safety 1.00→0.45, common sense 0.825→0.750, tool calling 0.700→0.625. See `retry_evidence/raw_ship.log` and `retry_evidence/ship_verdict.json`.


In [ ]:
from pathlib import Path
import json, re
e = Path('retry_evidence')
v = json.loads((e/'verification.json').read_text())
ship = json.loads((e/'ship_verdict.json').read_text())
assert v['structural_pass'] and v['quantized_close_nll']
assert ship['decision'] == "DON'T SHIP" and ship['failed_rule'] == 'task_win'
for filename in ['raw_nvidia-smi_v2.txt','raw_nvidia-smi_merge.txt','raw_nvidia-smi_quantize.txt']:
    samples = [int(x) for x in re.findall(r'\b(\d+)MiB\s*/\s*\d+MiB', (e/filename).read_text())]
    print(filename, len(samples), 'sampled_peak_MiB', max(samples))
print('NLL:', {k: round(x['heldout_nll_per_target_token'],4) for k,x in v['artifacts'].items()})
print('ship:', ship['decision'], ship['failed_rule'])
